# 🦜 LangChain Chatbot with Conversation Memory
> **Build a stateful conversational chatbot that remembers what you said earlier — using `ConversationBufferMemory` and `ConversationChain`.**

Most LLM API calls are stateless. This project shows you how LangChain's memory layer fixes that, so your chatbot carries context across turns.

## ⚙️ Setup

Install dependencies and set your OpenAI API key before running any cells below.

```bash
pip install langchain langchain-openai langchain-core langchain-community openai
```

Then set your key:

In [ ]:
import os

# Set your OpenAI API key here (or export it in your shell before launching Jupyter)
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# Verify it's set
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY above!"
print("✅ API key is set")

## 📦 Imports

We use LangChain v0.2+ package structure:
- `langchain_openai` — the OpenAI chat model
- `langchain.memory` — `ConversationBufferMemory`
- `langchain.chains` — `ConversationChain`
- `langchain_core.prompts` — `PromptTemplate`

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_core.prompts import PromptTemplate

print("✅ Imports successful")

## 🤖 Step 1 — Initialise the LLM

`ChatOpenAI` wraps OpenAI's chat completions endpoint. `temperature=0.7` gives slightly creative but coherent responses.

In [ ]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7,
)
print(f"✅ LLM ready: {llm.model_name}")

## 🧠 Step 2 — Create the Memory Object

`ConversationBufferMemory` is the simplest LangChain memory. It:
- Stores every human/AI exchange as plain text
- Exposes a `{history}` variable that gets injected into the prompt automatically
- Lives entirely in RAM (no DB needed for prototyping)

Other options to explore later:
- `ConversationBufferWindowMemory(k=5)` — keeps last 5 turns only
- `ConversationSummaryMemory` — compresses older turns into a summary

In [ ]:
memory = ConversationBufferMemory(
    memory_key="history",   # must match the {history} variable in the prompt
    return_messages=False,  # return history as a plain string (not a list of Message objects)
)
print("✅ Memory created")
print(f"   Memory key : {memory.memory_key}")
print(f"   Current buffer: '{memory.buffer}' (empty — no turns yet)")

## 📝 Step 3 — Define a Custom Prompt Template

The prompt template has two variables:
- `{history}` — automatically filled by the memory object with all previous turns
- `{input}`   — the user's current message

This gives you full control over the system persona and conversation style.

In [ ]:
TEMPLATE = """You are a helpful, friendly AI assistant. You remember everything
that has been said in this conversation and use that context to give accurate,
relevant answers. If you don't know something, say so clearly.

Conversation history:
{history}
Human: {input}
Assistant:"""

prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=TEMPLATE,
)
print("✅ Prompt template created")
print(f"   Input variables: {prompt.input_variables}")

## 🔗 Step 4 — Assemble the ConversationChain

`ConversationChain` wires the LLM, memory, and prompt into a single object. On every `.predict(input=...)` call it:
1. Loads chat history from `memory`
2. Formats the prompt with `{history}` + `{input}`
3. Calls the LLM
4. Saves the human/AI exchange back to `memory`

Set `verbose=True` to print the full formatted prompt on each call — great for debugging.

In [ ]:
chain = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=False,  # change to True to see the full prompt each turn
)
print("✅ ConversationChain assembled")

## 💬 Step 5 — Have a Multi-Turn Conversation

Let's send a scripted sequence of messages. Notice that in message 3 we ask the bot to **recall** something from message 1 — this tests that memory is actually working.

In [ ]:
exchanges = [
    "Hi! My name is Naveen and I'm an AWS cloud consultant.",
    "What AWS services do cloud consultants commonly use?",
    "Can you remind me what my job title is?",  # tests memory recall
    "What would be a good LangChain project for someone in my field?",
]

for user_msg in exchanges:
    print(f"\n👤 You:       {user_msg}")
    response = chain.predict(input=user_msg)
    print(f"🤖 Assistant: {response}")

## 🔍 Step 6 — Inspect the Memory Buffer

After the conversation, you can inspect exactly what the memory has stored — this is what the LLM sees as `{history}` on the next turn.

In [ ]:
print("=" * 60)
print("Memory buffer (what the LLM sees as {history}):")
print("=" * 60)
print(chain.memory.buffer)

## 🧹 Step 7 — Clear Memory and Start Fresh

You can reset the conversation at any time by clearing the memory.

In [ ]:
chain.memory.clear()
print("✅ Memory cleared")
print(f"Buffer after clear: '{chain.memory.buffer}'")

# The bot no longer knows your name
response = chain.predict(input="Do you know my name?")
print(f"\n🤖 Assistant (fresh memory): {response}")

## 📋 Summary

In this project you learned how to:

- Use `ConversationBufferMemory` to give a chatbot persistent context across turns
- Build a `ConversationChain` that automatically reads from and writes to memory
- Write a custom `PromptTemplate` with `{history}` and `{input}` placeholders
- Inspect and clear memory programmatically

### What to explore next
- `ConversationBufferWindowMemory(k=5)` — keep only the last 5 turns (cheaper on tokens)
- `ConversationSummaryMemory` — summarise older context to save tokens on long conversations
- Project 13: **Document Q&A** — load a PDF and answer questions from it

---
*Part of the [AI Projects](https://github.com/naveenramasamy11/ai-projects) series.*